# Notebook: entrenar YOLOX desde Roboflow, exportar a ONNX y descargar el modelo

Este notebook esta pensado para usuarios de Google Colab que tienen su dataset publicado en Roboflow y quieren entrenar un modelo YOLOX, exportarlo a ONNX y descargar el archivo final con las clases empaquetadas.


## Antes de empezar

Este notebook hace todo el flujo completo:

1. instala YOLOX y sus dependencias,
2. descarga el dataset desde Roboflow en formato COCO,
3. reorganiza el dataset dentro de Colab,
4. entrena el modelo,
5. exporta el mejor checkpoint a ONNX,
6. inserta los nombres de clase dentro del metadato del ONNX,
7. descarga el ONNX final.

Recomendaciones:

- Ejecuta las celdas en orden, de arriba hacia abajo.
- Usa un entorno de Colab con GPU activa.
- Verifica que tu dataset de Roboflow tenga split de `train` y `valid`.
- Usa una version ya publicada en Roboflow.


## Estructura esperada del dataset descargado desde Roboflow

Este notebook esta preparado para trabajar con la descarga de Roboflow en formato COCO, normalmente con una estructura parecida a esta:

```text
dataset/
|-- train/
|   |-- _annotations.coco.json
|   |-- img1.jpg
|   `-- img2.jpg
|-- valid/
|   |-- _annotations.coco.json
|   |-- img3.jpg
|   `-- img4.jpg
`-- test/
    |-- _annotations.coco.json
    `-- ...
```

Importante:

- `test/` es opcional y no se usa para entrenar.
- El notebook detecta automaticamente los JSON de `train` y `valid`.
- El numero de clases se obtiene desde `categories` del JSON COCO.
- El ONNX final guardara esos nombres de clase dentro del metadato `names`.


In [ ]:
# ======================================
# 1) CONFIGURACION GENERAL DEL NOTEBOOK
# ======================================
from pathlib import Path

BASE_DIR = Path("/content")
YOLOX_DIR = BASE_DIR / "YOLOX"
DATASET_RAW_DIR = BASE_DIR / "dataset_raw"
DATASET_DIR = BASE_DIR / "dataset_coco"
ANN_DIR = DATASET_DIR / "annotations"
EXP_FILE = YOLOX_DIR / "yolox_config.py"

YOLOX_VARIANT = "yolox_s"   # yolox_nano, yolox_tiny, yolox_s, yolox_m, yolox_l, yolox_x
BATCH_SIZE = 8
MAX_EPOCH = 50
# Recomendacion:
# - maximo 640x640 si el dataset es cuadrado
# - maximo 1024x576 si la relacion de aspecto es 16:9 horizontal
# - maximo 576x1024 si la relacion de aspecto es 16:9 vertical
# INPUT_WIDTH e INPUT_HEIGHT deben ser multiplos de 32 para YOLOX.
INPUT_WIDTH = 640
INPUT_HEIGHT = 640

if INPUT_WIDTH % 32 != 0 or INPUT_HEIGHT % 32 != 0:
    raise ValueError("INPUT_WIDTH e INPUT_HEIGHT deben ser multiplos de 32 para YOLOX")
NUM_WORKERS = 2
USE_FP16 = True

PRETRAINED_WEIGHTS = {
    "yolox_nano": "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth",
    "yolox_tiny": "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth",
    "yolox_s":    "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth",
    "yolox_m":    "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_m.pth",
    "yolox_l":    "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_l.pth",
    "yolox_x":    "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_x.pth",
}

MODEL_SCALES = {
    "yolox_nano": (0.33, 0.25),
    "yolox_tiny": (0.33, 0.375),
    "yolox_s":    (0.33, 0.50),
    "yolox_m":    (0.67, 0.75),
    "yolox_l":    (1.00, 1.00),
    "yolox_x":    (1.33, 1.25),
}

assert YOLOX_VARIANT in PRETRAINED_WEIGHTS, f"Variante no soportada: {YOLOX_VARIANT}"
depth, width = MODEL_SCALES[YOLOX_VARIANT]
PRETRAINED_PATH = BASE_DIR / f"{YOLOX_VARIANT}.pth"
PRETRAINED_URL = PRETRAINED_WEIGHTS[YOLOX_VARIANT]

print("Configuracion actual")
print("YOLOX_VARIANT =", YOLOX_VARIANT)
print("BATCH_SIZE    =", BATCH_SIZE)
print("MAX_EPOCH     =", MAX_EPOCH)
print("INPUT_WIDTH   =", INPUT_WIDTH)
print("INPUT_HEIGHT  =", INPUT_HEIGHT)
print("NUM_WORKERS   =", NUM_WORKERS)
print("USE_FP16      =", USE_FP16)
print("PRETRAINED    =", PRETRAINED_PATH)
print("depth,width   =", (depth, width))


In [ ]:
# =========================================
# 2) VERIFICAR QUE COLAB TENGA GPU REAL
# =========================================
import torch

print("torch.__version__ =", torch.__version__)
print("torch.cuda.is_available() =", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    raise RuntimeError(
        "Colab no tiene GPU activa. Ve a Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU."
    )


In [ ]:
# ================================================
# 3) INSTALAR DEPENDENCIAS, YOLOX Y ROBOFLOW
# ================================================
%cd /content

import re
import shutil

!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q pycocotools onnx onnxscript onnxruntime loguru tabulate thop ninja roboflow "jedi>=0.16"

if YOLOX_DIR.exists():
    shutil.rmtree(YOLOX_DIR)
    print("YOLOX anterior eliminado")

!git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

req_file = YOLOX_DIR / "requirements.txt"
txt = req_file.read_text(encoding="utf-8")
txt = re.sub(r'^.*onnx-simplifier.*\n?', '', txt, flags=re.MULTILINE)
txt = re.sub(r'^.*onnxsim.*\n?', '', txt, flags=re.MULTILINE)
req_file.write_text(txt, encoding="utf-8")

!python -m pip install -q -r /content/YOLOX/requirements.txt

print("Instalacion terminada.")


In [ ]:
# ======================================================
# 4) FORZAR PYTHONPATH Y VALIDAR IMPORTACION DE YOLOX
# ======================================================
import os
import sys

if str(YOLOX_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOX_DIR))

current_pythonpath = os.environ.get("PYTHONPATH", "")
paths = [p for p in current_pythonpath.split(":") if p] if current_pythonpath else []
if str(YOLOX_DIR) not in paths:
    os.environ["PYTHONPATH"] = f"{YOLOX_DIR}:{current_pythonpath}" if current_pythonpath else str(YOLOX_DIR)

import yolox
print("PYTHONPATH =", os.environ.get("PYTHONPATH"))
print("Importacion OK de yolox desde:", yolox.__file__)


In [ ]:
# ============================================
# 5) DESCARGAR EL PESO BASE PREENTRENADO
# ============================================
if not PRETRAINED_PATH.exists():
    !wget -O {PRETRAINED_PATH} {PRETRAINED_URL}
else:
    print("Ya existe:", PRETRAINED_PATH)

print("Peso base listo en:", PRETRAINED_PATH)


## Configuracion de Roboflow

En la siguiente celda solo debes completar 2 datos:

- `ROBOFLOW_API_KEY`: tu API key personal.
- `ROBOFLOW_DATASET_URL`: la URL de la version del dataset que vas a usar.

Como obtenerlos:

- La `API Key` la encuentras en tu cuenta de Roboflow.
- La URL se obtiene entrando a la version publicada del dataset en Roboflow y copiando la direccion del navegador.

Ejemplo de URL de una version:

`https://app.roboflow.com/mi-workspace/mi-proyecto/3`

A partir de esa URL, este notebook detecta automaticamente:

- `ROBOFLOW_WORKSPACE = "mi-workspace"`
- `ROBOFLOW_PROJECT = "mi-proyecto"`
- `ROBOFLOW_VERSION = 3`

Importante:

- Esos valores salen de la URL; no tienes que volver a escribirlos manualmente.
- Solo pega una vez tu `API Key` y una vez la URL de la version del dataset.
- Si la URL no termina con el numero de version, el notebook te avisara con un mensaje claro.


In [ ]:
# ==============================================
# 6) DESCARGAR DATASET DESDE ROBOFLOW EN COCO
# ==============================================
from roboflow import Roboflow
from urllib.parse import urlparse
import shutil

ROBOFLOW_API_KEY = "PEGA_AQUI_TU_API_KEY"
ROBOFLOW_DATASET_URL = "PEGA_AQUI_TU_DATASET_URL"

if not ROBOFLOW_API_KEY.strip() or "PEGA_AQUI" in ROBOFLOW_API_KEY:
    raise ValueError("Completa ROBOFLOW_API_KEY con tu clave real de Roboflow.")
if not ROBOFLOW_DATASET_URL.strip() or "app.roboflow.com" not in ROBOFLOW_DATASET_URL:
    raise ValueError("Completa ROBOFLOW_DATASET_URL con la URL completa de la version del dataset en Roboflow.")

parsed_url = urlparse(ROBOFLOW_DATASET_URL.strip())
url_parts = [part for part in parsed_url.path.split("/") if part]

if len(url_parts) < 3:
    raise ValueError("La URL no tiene el formato esperado. Debe verse como https://app.roboflow.com/workspace/proyecto/version")

ROBOFLOW_WORKSPACE = url_parts[0]
ROBOFLOW_PROJECT = url_parts[1]

try:
    ROBOFLOW_VERSION = int(url_parts[2])
except ValueError as exc:
    raise ValueError("No se pudo detectar ROBOFLOW_VERSION desde la URL. Verifica que la URL termine con el numero de version.") from exc

print("Workspace detectado :", ROBOFLOW_WORKSPACE)
print("Project detectado   :", ROBOFLOW_PROJECT)
print("Version detectada   :", ROBOFLOW_VERSION)

for p in [DATASET_RAW_DIR, DATASET_DIR]:
    if p.exists():
        shutil.rmtree(p)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
downloaded = version.download("coco")

DATASET_RAW_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(downloaded.location, DATASET_RAW_DIR, dirs_exist_ok=True)

print("Dataset descargado desde Roboflow en:", downloaded.location)
print("Dataset copiado a:", DATASET_RAW_DIR)


In [ ]:
# ===========================================
# 7) REORGANIZAR EL DATASET PARA YOLOX
# ===========================================
import json

DATASET_DIR.mkdir(parents=True, exist_ok=True)
ANN_DIR.mkdir(parents=True, exist_ok=True)
(DATASET_DIR / "train2017").mkdir(parents=True, exist_ok=True)
(DATASET_DIR / "val2017").mkdir(parents=True, exist_ok=True)

json_files = list(DATASET_RAW_DIR.rglob("*.json"))
image_files = []
for ext in ("*.jpg", "*.jpeg", "*.png"):
    image_files.extend(DATASET_RAW_DIR.rglob(ext))

if not json_files:
    raise FileNotFoundError("No se encontraron archivos .json en el dataset descargado")
if not image_files:
    raise FileNotFoundError("No se encontraron imagenes en el dataset descargado")

print("JSON encontrados:")
for jf in json_files:
    print("-", jf.relative_to(DATASET_RAW_DIR))

train_json = None
val_json = None

for jf in json_files:
    rel_path = str(jf.relative_to(DATASET_RAW_DIR)).lower().replace("\\", "/")
    name = jf.name.lower()

    if train_json is None and ("/train/" in f"/{rel_path}" or "train" in name):
        train_json = jf
    elif val_json is None and (("/valid/" in f"/{rel_path}") or ("/val/" in f"/{rel_path}") or ("valid" in name) or ("val" in name)):
        val_json = jf

if train_json is None:
    train_json = json_files[0]
    print("Aviso: no se detecto un JSON de train por nombre o carpeta. Se usara el primer JSON encontrado.")

if val_json is None:
    val_json = train_json
    print("Aviso: no se detecto un JSON de validacion. Se reutilizara el JSON de train.")

for img in image_files:
    shutil.copy2(img, DATASET_DIR / "train2017" / img.name)
    shutil.copy2(img, DATASET_DIR / "val2017" / img.name)

shutil.copy2(train_json, ANN_DIR / "instances_train.json")
shutil.copy2(val_json, ANN_DIR / "instances_val.json")

print("Dataset reorganizado en:", DATASET_DIR)
print("JSON train usado:", train_json.relative_to(DATASET_RAW_DIR))
print("JSON val usado  :", val_json.relative_to(DATASET_RAW_DIR))


In [ ]:
# ==================================
# 8) VALIDAR ESTRUCTURA DEL DATASET
# ==================================
train_imgs = list((DATASET_DIR / "train2017").glob("*"))
val_imgs = list((DATASET_DIR / "val2017").glob("*"))

print("Imagenes en train2017:", len(train_imgs))
print("Imagenes en val2017  :", len(val_imgs))
print("Nota: estas carpetas son una reorganizacion interna para YOLOX.")
print("La asignacion real de train y val depende del contenido de los JSON COCO detectados arriba.")

for jf in [ANN_DIR / "instances_train.json", ANN_DIR / "instances_val.json"]:
    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("\nArchivo:", jf.name)
    print("images     =", len(data.get("images", [])))
    print("annotations=", len(data.get("annotations", [])))
    print("categories =", len(data.get("categories", [])))

    if not data.get("images"):
        raise ValueError(f"{jf.name} no contiene la clave 'images' o esta vacia")
    if not data.get("categories"):
        raise ValueError(f"{jf.name} no contiene la clave 'categories' o esta vacia")


In [ ]:
# =======================================
# 9) CREAR yolox_config.py DINAMICO
# =======================================
with open(ANN_DIR / "instances_train.json", "r", encoding="utf-8") as f:
    coco_train = json.load(f)

num_classes = len(coco_train.get("categories", []))
if num_classes <= 0:
    raise ValueError("No se detectaron categorias en instances_train.json")

config_text = f'''
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        self.num_classes = {num_classes}
        self.depth = {depth}
        self.width = {width}
        self.data_dir = "{DATASET_DIR}"
        self.train_ann = "instances_train.json"
        self.val_ann = "instances_val.json"
        self.input_size = ({INPUT_HEIGHT}, {INPUT_WIDTH})
        self.test_size = ({INPUT_HEIGHT}, {INPUT_WIDTH})
        self.multiscale_range = 0
        self.random_size = ({INPUT_HEIGHT // 32}, {INPUT_HEIGHT // 32})
        self.max_epoch = {MAX_EPOCH}
        self.data_num_workers = {NUM_WORKERS}
        self.eval_interval = 1
        self.print_interval = 10
        self.basic_lr_per_img = 0.01 / 64.0
        self.warmup_epochs = 1
        self.no_aug_epochs = 1
        self.enable_mixup = False
        self.mosaic_prob = 1.0
        self.mixup_prob = 0.0
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.exp_name = "yolox_custom_coco"

def get_exp():
    return Exp()
'''

EXP_FILE.write_text(config_text, encoding="utf-8")
print("Archivo creado:", EXP_FILE)
print(EXP_FILE.read_text(encoding="utf-8"))


In [ ]:
# ======================================
# 10) ENTRENAR EL MODELO EN YOLOX
# ======================================
%cd /content/YOLOX

if not torch.cuda.is_available():
    raise RuntimeError(
        "Torch no tiene CUDA disponible. Revisa que Colab tenga GPU activa y reinicia el entorno."
    )

os.environ["PYTHONPATH"] = f"/content/YOLOX:" + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH usado para entrenamiento:", os.environ["PYTHONPATH"])
print("torch.cuda.is_available() =", torch.cuda.is_available())
print("GPU actual:", torch.cuda.get_device_name(0))

train_cmd = f"PYTHONPATH=/content/YOLOX:$PYTHONPATH python tools/train.py -f {EXP_FILE} -d 1 -b {BATCH_SIZE} -o "
if USE_FP16:
    train_cmd += f"--fp16 -c {PRETRAINED_PATH}"
else:
    train_cmd += f"-c {PRETRAINED_PATH}"

print("Comando de entrenamiento:")
print(train_cmd)

!{train_cmd}


In [ ]:
# ==========================================
# 11) ENCONTRAR EL CHECKPOINT ENTRENADO
# ==========================================
output_dir = BASE_DIR / "YOLOX" / "YOLOX_outputs" / "yolox_custom_coco"

candidates = [
    output_dir / "best_ckpt.pth",
    output_dir / "latest_ckpt.pth",
]

trained_ckpt = None
for c in candidates:
    if c.exists():
        trained_ckpt = c
        break

if trained_ckpt is None:
    all_pth = list(output_dir.rglob("*.pth")) if output_dir.exists() else []
    raise FileNotFoundError(f"No se encontro checkpoint entrenado en {output_dir}. Encontrados: {all_pth}")

print("Checkpoint encontrado:", trained_ckpt)


In [ ]:
# ==========================================================
# 12) VALIDAR DEPENDENCIAS NECESARIAS ANTES DE EXPORTAR A ONNX
# ==========================================================
import onnx
import onnxscript
import onnxruntime

print("onnx       =", onnx.__version__)
print("onnxscript =", onnxscript.__version__)
print("onnxruntime=", onnxruntime.__version__)


In [ ]:
# =========================================================
# 13) PARCHEAR tools/export_onnx.py PARA COLAB ACTUAL
# =========================================================
export_file = BASE_DIR / "YOLOX" / "tools" / "export_onnx.py"
txt = export_file.read_text(encoding="utf-8")

txt = txt.replace(
    'ckpt = torch.load(ckpt_file, map_location="cpu")',
    'ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)'
)

txt = txt.replace(
    "torch.onnx._export(",
    "torch.onnx.export("
)

export_file.write_text(txt, encoding="utf-8")
print("Parche aplicado en:", export_file)


In [ ]:
# =========================================
# 14) EXPORTAR EL CHECKPOINT A ONNX
# =========================================
%cd /content/YOLOX

import subprocess

onnx_path = output_dir / "best_ckpt_empaquetado.onnx"

cmd = [
    "python", "tools/export_onnx.py",
    "-f", str(EXP_FILE),
    "-c", str(trained_ckpt),
    "--output-name", str(onnx_path),
    "--no-onnxsim",
    "--decode_in_inference",
]

env = os.environ.copy()
env["PYTHONPATH"] = "/content/YOLOX:" + env.get("PYTHONPATH", "")

print("Comando real:")
print("PYTHONPATH=" + env["PYTHONPATH"])
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    cwd="/content/YOLOX",
    env=env,
    capture_output=True,
    text=True
)

print("\n===== STDOUT =====")
print(result.stdout if result.stdout else "(vacio)")

print("\n===== STDERR =====")
print(result.stderr if result.stderr else "(vacio)")

print("\nCodigo de salida:", result.returncode)
print("Existe el ONNX?:", onnx_path.exists(), onnx_path)

if not onnx_path.exists():
    raise FileNotFoundError(f"No se genero el ONNX en {onnx_path}")

print("Modelo ONNX generado en:", onnx_path)


In [ ]:
# ===================================================
# 15) INSERTAR LAS CLASES EN EL METADATO DEL ONNX
# ===================================================
import uuid

random_id = uuid.uuid4().hex[:8]
onnx_in = output_dir / "best_ckpt_empaquetado.onnx"
onnx_out = output_dir / f"best_ckpt_{random_id}.onnx"

with open(ANN_DIR / "instances_train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

class_dict = {idx: cat["name"] for idx, cat in enumerate(data.get("categories", []))}

model = onnx.load(str(onnx_in))
kept = [p for p in model.metadata_props if p.key != "names"]
del model.metadata_props[:]
model.metadata_props.extend(kept)

model.metadata_props.append(
    onnx.StringStringEntryProto(
        key="names",
        value=json.dumps(class_dict, ensure_ascii=False)
    )
)

onnx.save(model, str(onnx_out))

print("ONNX final con metadato 'names' guardado en:", onnx_out)
print("Las clases embebidas en el modelo son:", class_dict)


In [ ]:
# ============================================
# 16) VERIFICAR METADATO DEL ONNX GENERADO
# ============================================
m = onnx.load(str(onnx_out))
meta = {p.key: p.value for p in m.metadata_props}

print("Metadata encontrada:")
for k, v in meta.items():
    print(f"{k} = {v}")

if "names" not in meta:
    raise KeyError("No se encontro el metadato 'names' en el ONNX final")

names = json.loads(meta["names"])
print("\nClases recuperadas desde metadata:")
print(names)


## Nota

El ONNX final generado por este notebook queda pensado para una inferencia donde la salida ya incluye la decodificacion dentro del modelo exportado.

- Incluye el metadato `names` con los nombres de clase detectados desde `categories`.
- Si luego cambias tu pipeline de inferencia, revisa si tu codigo espera salida decodificada o salida cruda del modelo.
- Si usas otro script distinto para ONNX, valida primero que no aplique un postproceso incompatible.


In [ ]:
# ==================================
# 17) DESCARGAR SOLO EL ONNX FINAL
# ==================================
from google.colab import files

if not onnx_out.exists():
    raise FileNotFoundError(f"No existe el archivo final esperado: {onnx_out}")

print("Descargando:", onnx_out)
print("Este es el archivo final que debes conservar y usar fuera de Colab.")
files.download(str(onnx_out))
